<a href="https://colab.research.google.com/github/AdrionRosanelli/LoRa_Sionna/blob/main/Teste_LoRa_PHY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Teste de Implementação do LoRa PHY no Sionna.**

Implementação realizada a seguir utilizando funções do .py:

# Implementação do transmissor da camada física do LoRa


### Imports

Instalação e importação das bibliotecas. (Import do tutorial Part 1 Sionna PHY)

In [ ]:
import os # Configure which GPU
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import Sionna
try:
    import sionna.phy
except ImportError as e:
    import sys
    if 'google.colab' in sys.modules:
       # Install Sionna in Google Colab
       print("Installing Sionna and restarting the runtime. Please run the cell again.")
       os.system("pip install sionna")
       os.kill(os.getpid(), 5)
    else:
       raise e

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

import numpy as np

# For plotting
%matplotlib inline
# also try %matplotlib widget

import matplotlib.pyplot as plt

# for performance measurements
import time

In [ ]:
from sionna.phy.channel import OFDMChannel, RayleighBlockFading
from sionna.phy.mimo import StreamManagement
from sionna.phy.ofdm import ResourceGrid, ResourceGridMapper, LSChannelEstimator, LMMSEEqualizer
from sionna.phy.fec.ldpc import LDPC5GEncoder, LDPC5GDecoder
from sionna.phy.mapping import Mapper, Demapper, Constellation, BinarySource
from sionna.phy.utils import ebnodb2no, sim_ber
import sionna

### Testes e Simulação do modelo

Inicialização e configuração da LoRa PHY:

In [ ]:
print("=== Simulação da Camada Física LoRa com Sionna ===\n")

# Inicializa LoRa PHY
lora_phy = LoRaPhyTransmitter(
        spreading_factor=7,
        bandwidth=125e3,
        coding_rate=1
)
print(f"Configurações LoRa:")
print(f"- Spreading Factor: {lora_phy.sf}")
print(f"- Largura de Banda: {lora_phy.bw/1000:.0f} kHz")
print(f"- Chips por símbolo: {lora_phy.n_chips}")
print(f"- Duração do símbolo: {lora_phy.symbol_duration*1000:.2f} ms")
print(f"- Taxa de símbolo: {1/lora_phy.symbol_duration:.2f} símbolos/s\n")

Análise da frequência pelo tempo:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

symbol_values = [0, 32, 64, 96]

for i, symbol_val in enumerate(symbol_values):
    chirp, t = lora_phy.modulate_symbol(symbol_val)
    freq_inst = lora_phy.calculate_instantaneous_frequency(chirp)
    t_freq = t[:-1]  # Tempo para frequência instantânea (um ponto a menos)

    # Plota parte real
    axes[i].plot(t_freq*1000, freq_inst/1000)
    axes[i].set_title(f'Frequência Instantânea - Símbolo {symbol_val}')
    axes[i].set_xlabel('Tempo (ms)')
    axes[i].set_ylabel('Frequência (kHz)')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Geração de dados e pacotes:

In [ ]:
# Dados para transmissão
payload = [1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1] * 4  # 64 bits
print(f"Payload: {len(payload)} bits")

# Gera pacote LoRa
packet, temp = lora_phy.generate_lora_packet(payload)
print(f"Pacote gerado: {len(packet)} amostras\n")
print(f"Duração total do pacote: {len(packet) * lora_phy.chip_duration * 1000:.2f} ms")


freq_inst = lora_phy.calculate_instantaneous_frequency(packet)
#t_freq = temp[:-1]  # Tempo para frequência instantânea (um ponto a menos)
Tempo = np.linspace(0, len(packet) * lora_phy.chip_duration, len(freq_inst));

plt.figure(figsize=(12, 4))
# Plota parte real
#plt.plot(t_freq*1000, freq_inst/1000)
plt.plot(Tempo*1000, freq_inst/1000, '-o', markersize=2)
plt.title(f'Frequência Instantânea')
plt.xlabel('Tempo (ms)')
plt.ylabel('Frequência (kHz)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()